## A1 - So sánh CNN (ResNet-50) và Transformer (ViT-B/16) trên Food-101

- Dataset lựa chọn: **Food-101**
  - 101,000 ảnh màu, 101 lớp món ăn
  - Split chính thức: 750 ảnh train + 250 ảnh test mỗi lớp
  - Ảnh train có nhiễu (một số nhãn sai), ảnh test được kiểm tra thủ công

### Chuẩn bị dữ liệu và DataLoader

Dataset quá lớn để nạp hết vào bộ nhớ (~5 GB JPEG), nên chỉ giữ danh sách `(image_id, label)` và **đọc + decode từng ảnh** khi DataLoader cần.

In [ ]:
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import math

from torch import nn
from torchvision.models import resnet50, vit_b_16

sys.path.append(str(Path.cwd().parent))  # repo root -> `common`

from common.utils import set_seed, get_device
from common.plotting import setup_style, save_fig
from data import (
    SPLITS, read_classes, load_or_build_manifest, file_sha256, check_manifest,
    make_datasets, make_loaders, denormalize,
)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| TorchVision:", torchvision.__version__, "| NumPy:", np.__version__)

In [ ]:
CONFIG = {
    "dataset": "Food-101",
    "data_dir": Path("../../food-101"),
    "manifest_path": Path("split_manifest.csv"),
    "split_seed": 2026,      # per-class RNG: default_rng(split_seed + class_index)
    "per_class": {"train": 600, "val": 150, "test": 250},
    "image_size": 224,
    "physical_batch": 32,
    "accum_steps": 4,            # effective batch = 32 x 4 = 128
    "num_workers": 8,
    "seed": 42,
    "epochs": 10,  # số epoch cho train full dataset, chọn 10 vì thời gian quá lớn nếu epoch lớn
    "warmup_epochs": 1,  # 10% của epoch
    "lr_head": 3e-4,
    "backbone_lr_mult": {"scratch": 1.0, "pretrained": 0.1},
    "weight_decay": 0.01,
    "betas": (0.9, 0.999),
    "eps": 1e-8,
    "grad_clip": 1.0,
    "head_init_std": 0.01,
    "amp": True,                # FP16 autocast; dùng chung cho mọi cấu hình
}

EXPERIMENTS = [
    # (arch,      init,         freeze)
    ("resnet50", "scratch",    "full"),
    ("resnet50", "pretrained", "full"),
    ("vit_b_16", "pretrained", "full"),
    ("resnet50", "pretrained", "head"),     
    ("resnet50", "pretrained", "partial"),  
]

TRAINABLE_MODULES = {
    "resnet50": {
        "full":    None,                 # None = train toàn bộ
        "partial": ["layer4", "fc"],     # freeze conv1, bn1, layer1-3
        "head":    ["fc"],               # freeze toàn bộ backbone
    },
    "vit_b_16": {
        "full":    None,                 # phạm vi đã chốt chỉ dùng ViT full
    },
}

set_seed(CONFIG["seed"])
DEVICE = get_device()
FIG_DIR = Path("figures")
setup_style()

assert (CONFIG["data_dir"] / "images").is_dir(), f"Food-101 not found at {CONFIG['data_dir'].resolve()}"
print("Device:", DEVICE)
print("Effective batch:", CONFIG["physical_batch"] * CONFIG["accum_steps"])

In [ ]:
# Danh sách lớp: sắp xếp theo thứ tự chữ cái, chỉ số lớp = vị trí trong danh sách.

CLASS_NAMES = read_classes(CONFIG["data_dir"])
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SHAPE = (3, CONFIG["image_size"], CONFIG["image_size"])  # (C, H, W)

print("Number of classes:", NUM_CLASSES)
print("Image shape:", IMAGE_SHAPE)
print("First 5 classes:", CLASS_NAMES[:5])

In [ ]:
LOG_FILE = Path("results/experiments/train.log")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

def log(msg: str) -> None:
    line = f"[{time.strftime('%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

#### Chia tập dữ liệu và manifest

- Validation chỉ lấy từ **split train chính thức**: mỗi lớp sắp xếp ID, hoán vị bằng `default_rng(2026 + class_index)`, 150 ảnh đầu vào validation, 600 ảnh còn lại vào train.
- Test giữ nguyên split chính thức (250 ảnh/lớp).
- Manifest lưu `image_id, class_index, split, sha256, width, height, mode`. Lần đầu chạy sẽ đọc toàn bộ ảnh để hash và kiểm tra decode (vài phút); các lần sau dùng lại file đã lưu.

| Partition | Ảnh/lớp | Tổng |
|-|-:|-:|
| Train | 600 | 60,600 |
| Validation | 150 | 15,150 |
| Test | 250 | 25,250 |

In [ ]:
t0 = time.time()
manifest = load_or_build_manifest(
    CONFIG["manifest_path"], CONFIG["data_dir"], CLASS_NAMES,
    val_per_class=CONFIG["per_class"]["val"], split_seed=CONFIG["split_seed"],
)
print(f"Manifest rows: {len(manifest):,}  ({time.time() - t0:.1f}s)")
print("Manifest SHA-256:", file_sha256(CONFIG["manifest_path"]))

In [ ]:
# Kích thước ảnh gốc (đọc từ manifest, không cần mở ảnh): min / median / max của width và height,
# và số ảnh có cạnh ngắn < 224 (các ảnh này sẽ bị phóng to khi resize/crop).

widths = np.array([r["width"] for r in manifest if r["width"] > 0])
heights = np.array([r["height"] for r in manifest if r["height"] > 0])
short_edge = np.minimum(widths, heights)

print(f"Width : min {widths.min()}, median {np.median(widths):.0f}, max {widths.max()}")
print(f"Height: min {heights.min()}, median {np.median(heights):.0f}, max {heights.max()}")
print(f"Short edge < 224: {(short_edge < 224).sum()} images")

#### Tiền xử lý và augmentation

Tất cả cấu hình dùng chung một pipeline:

- **Train:** `RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.75, 4/3))` → `RandomHorizontalFlip(0.5)`
- **Validation / Test:** resize cạnh ngắn về 256 → `CenterCrop(224)`
- Nội suy bilinear có antialias, đưa pixel về $[0, 1]$, chuẩn hoá theo mean/std của ImageNet
- Không dùng TTA, MixUp, CutMix hay label smoothing

In [ ]:
datasets = make_datasets(manifest, CONFIG["data_dir"], CONFIG["image_size"])
loaders = make_loaders(
    datasets,
    batch_size=CONFIG["physical_batch"],
    seed=CONFIG["seed"],
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
)

for split in SPLITS:
    print(f"{split:5s}: {len(datasets[split]):6,} images, {len(loaders[split]):5,} batches")

In [ ]:
# Đo tốc độ đọc dữ liệu (decode JPEG + augmentation trên CPU workers).
# Nếu tốc độ này thấp hơn tốc độ GPU/MPS xử lý thì DataLoader là nút thắt cổ chai -> tăng num_workers.

n_batches = 50
it = iter(loaders["train"])
next(it)  # warm up workers
t0 = time.time()
for _ in range(n_batches):
    next(it)
elapsed = time.time() - t0
print(f"Train loader: {n_batches * CONFIG['physical_batch'] / elapsed:.0f} images/s "
      f"({elapsed / n_batches * 1000:.0f} ms/batch, {CONFIG['num_workers']} workers)")

In [ ]:
# Biểu đồ số ảnh mỗi lớp theo split (cột chồng): xác nhận dữ liệu cân bằng 600 / 150 / 250 ảnh mỗi lớp.

fig, ax = plt.subplots(figsize=(10, 3))
x = np.arange(NUM_CLASSES)
bottom = np.zeros(NUM_CLASSES)
for split, color in zip(SPLITS, ["#2a78d6", "#104281", "#c9c8c3"]):
    counts = Counter(datasets[split].labels)
    heights_ = np.array([counts[c] for c in range(NUM_CLASSES)])
    ax.bar(x, heights_, bottom=bottom, color=color, label=split, width=0.9)
    bottom += heights_
ax.set_xlabel("Class index")
ax.set_ylabel("Images")
ax.set_title("Food-101: images per class and split")
ax.legend(frameon=False, ncol=3)
save_fig(fig, FIG_DIR, "class_distribution", show=True)

In [ ]:
# show_samples(dataset, n_classes, cols, title)
# - Lấy ảnh đầu tiên của các lớp cách đều nhau, hiển thị sau transform của val.
# - `denormalize` đưa tensor đã chuẩn hoá ImageNet về [0, 1] để imshow hiển thị đúng màu.

def show_samples(dataset, n_classes=16, cols=8, title=""):
    first_of_class = {}
    for i, y in enumerate(dataset.labels):
        first_of_class.setdefault(y, i)
    idx = [first_of_class[c] for c in range(0, NUM_CLASSES, NUM_CLASSES // n_classes)][:n_classes]
    rows = int(np.ceil(len(idx) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.8))
    for ax, i in zip(axes.flat, idx):
        img, y = dataset[i]
        ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
        ax.set_title(CLASS_NAMES[y].replace("_", " "), fontsize=7)
        ax.axis("off")
    for ax in axes.flat[len(idx):]:
        ax.axis("off")
    fig.suptitle(title)
    return fig

fig = show_samples(datasets["val"], title="Validation samples (resize 256 → center crop 224)")
save_fig(fig, FIG_DIR, "samples_val", show=True)

In [ ]:
# Minh hoạ augmentation: gọi lại cùng một ảnh train nhiều lần, mỗi lần transform ngẫu nhiên cho kết quả khác
# (RandomResizedCrop scale 0.8-1.0 + lật ngang với xác suất 0.5).

img_idx = 0
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax in axes:
    img, y = datasets["train"][img_idx]
    ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle(f"Train augmentation: {CLASS_NAMES[y].replace('_', ' ')}")
save_fig(fig, FIG_DIR, "augmentation_preview", show=True)

## classifiers: ResNet-50, ViT-B/16

In [ ]:
from torchvision.models import ResNet50_Weights, ViT_B_16_Weights

PRETRAINED_WEIGHTS = {
    "resnet50": ResNet50_Weights.IMAGENET1K_V1,   # top-1 ImageNet 76.1%
    "vit_b_16": ViT_B_16_Weights.IMAGENET1K_V1,   # top-1 ImageNet 81.1%
}
HEAD_PREFIX = {"resnet50": "fc.", "vit_b_16": "heads."}


def build_model(arch: str, num_classes: int, init: str = "scratch",
                head_std: float = 0.01, head_seed: int = 0) -> nn.Module:
    if init not in ("scratch", "pretrained"):
        raise ValueError(f"Unsupported init: {init}")

    weights = PRETRAINED_WEIGHTS[arch] if init == "pretrained" else None
    
    if arch == "resnet50":
        model = resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes) # 2048 -> 101
        head = model.fc

    elif arch == "vit_b_16":
        model = vit_b_16(weights=weights)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes) # 768 -> 101
        head = model.heads.head
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    # Head khởi tạo bằng RNG riêng -> giống hệt nhau giữa scratch và pretrained cùng seed
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(head_seed)
        nn.init.normal_(head.weight, std=head_std)
        nn.init.zeros_(head.bias)
        
    return model

In [ ]:
def apply_freeze(model: nn.Module, arch: str, freeze: str) -> list[str]:
    """Đặt requires_grad theo chế độ freeze; trả về tên các module con bị freeze hoàn toàn."""
    modes = TRAINABLE_MODULES[arch]
    if freeze not in modes:
        raise ValueError(f"Freeze mode '{freeze}' is not defined for {arch}")
    keep = modes[freeze]

    def is_trainable(name: str) -> bool:
        return keep is None or any(name == k or name.startswith(k + ".") for k in keep)

    for name, p in model.named_parameters():
        p.requires_grad = is_trainable(name)

    # Module cấp cao nhất bị freeze toàn bộ -> sẽ được giữ ở eval() khi train
    frozen = [name for name, _ in model.named_children() if not is_trainable(name)]
    model.frozen_modules = frozen
    return frozen

In [ ]:
def param_groups(model, arch: str, lr_head: float, backbone_mult: float, weight_decay: float):
    groups = {(part, decay): [] for part in ("backbone", "head") for decay in (True, False)}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        part = "head" if name.startswith(HEAD_PREFIX[arch]) else "backbone"
        no_decay = p.ndim <= 1 or "pos_embedding" in name or "class_token" in name
        groups[(part, not no_decay)].append(p)

    lr = {"head": lr_head, "backbone": lr_head * backbone_mult}
    
    return [
        {"name": f"{part}_{'decay' if decay else 'no_decay'}", "params": ps,
         "lr": lr[part], "weight_decay": weight_decay if decay else 0.0}
        for (part, decay), ps in groups.items() if ps          # bỏ nhóm rỗng (ví dụ head-only)
    ]

In [ ]:
def make_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:                                   # 0.1x -> 1x
            return 0.1 + 0.9 * step / max(1, warmup_steps)
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * t))   # 1x -> 0.01x
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
def set_train_mode(model: nn.Module) -> None:
    """model.train() cho phần trainable, nhưng giữ mọi module bị freeze ở eval()."""
    model.train()
    for name in getattr(model, "frozen_modules", []):
        model.get_submodule(name).eval()

In [ ]:
import torch.nn.functional as F

def accumulation_group_sizes(n_samples: int, batch_size: int, accum_steps: int) -> list[int]:
    """Số mẫu thật trong từng nhóm accumulation (nhóm = tối đa `accum_steps` batch liên tiếp).

    Ví dụ Food-101 train: 60,600 ảnh, batch 32, accum 4
      -> 1,894 batch (batch cuối 24 ảnh) -> 474 nhóm, nhóm cuối gồm 2 batch = 56 ảnh.
    """

    n_batches = -(-n_samples // batch_size)  # ceil division
    batch_sizes = [min(batch_size, n_samples - b * batch_size) for b in range(n_batches)]

    group_sizes = [sum(batch_sizes[i:i+accum_steps]) for i in range(0, n_batches, accum_steps)]
    return group_sizes

def train_one_epoch(model, loader, optimizer, scheduler, scaler, device, 
                    accum_steps: int, clip: float, amp: bool = True,
                    log_every: int = 200, tag: str = ""):
    """Một epoch train với gradient accumulation + AMP + gradient clipping.
    
    Args:
        model: Mô hình PyTorch.
        loader: DataLoader cho dữ liệu train.
        optimizer: Optimizer.
        scheduler: Learning rate scheduler.
        scaler: GradScaler cho AMP.
        device: Thiết bị (CPU/GPU).
        accum_steps: Số bước để tích lũy gradient.
        clip: Giá trị để clip gradient.
        amp: Sử dụng AMP hay không.
    """

    # model.train()
    set_train_mode(model)
    group_sizes = accumulation_group_sizes(len(loader.dataset), loader.batch_size, accum_steps)
    n_batches = len(loader)
    trainable = [p  for p in model.parameters() if p.requires_grad]

    # Cộng dồn trên device để tránh đồng bộ GPU/MPS ở mỗi batch (.item() rất chậm)
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device, dtype=torch.long)
    seen = 0

    optimizer.zero_grad(set_to_none=True)
    t_start = time.time()

    for i, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        n_group = group_sizes[i // accum_steps]

        # 1) Forward trong autocast (FP16) để tiết kiệm memory/thời gian
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(x) # [B, 101]

        batch_loss = F.cross_entropy(logits.float(), y, reduction="sum")   # loss tính ở FP32 cho ổn định

        # 2) Backward: chia cho số mẫu của CẢ NHÓM, gradient được cộng dồn qua các batch
        scaler.scale(batch_loss / n_group).backward()

        # Thống kê (không cần gradient)
        loss_sum += batch_loss.detach()
        correct += (logits.argmax(dim=1) == y).sum()
        seen += y.size(0)

        # 3) Hết nhóm (đủ accum_steps batch, hoặc batch cuối epoch) -> một optimizer step
        end_of_group = (i + 1) % accum_steps == 0 or (i + 1) == n_batches
        if end_of_group:
            scaler.unscale_(optimizer)                                     # gradient về đúng thang đo thật
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=clip)       # clip trên gradient thật
            scaler.step(optimizer)                                         # bỏ qua step nếu có inf/NaN
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()                                               # LR theo optimizer step

        # Log thơng tin
        if log_every and ((i + 1) % log_every == 0 or i + 1 == n_batches):
            done = (i + 1) / n_batches
            elapsed = time.time() - t_start
            print(f"  {tag} batch {i + 1:5d}/{n_batches} ({done:5.1%}) | "
                  f"loss {(loss_sum / seen).item():.3f} | "
                  f"{elapsed / 60:5.1f} min, ETA {elapsed / done * (1 - done) / 60:5.1f} min", flush=True)

    return {
        "loss": (loss_sum / seen).item(), 
        "acc": (correct.float() / seen).item()
        }


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, amp: bool = True, keep_outputs: bool = False):
    model.eval()
    loss_sum = torch.zeros((), device=device)
    top1 = torch.zeros((), device=device, dtype=torch.long)
    top5 = torch.zeros((), device=device, dtype=torch.long)
    seen = 0
    all_y, all_logits = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(x)

        logits = logits.float()

        loss_sum += F.cross_entropy(logits, y, reduction="sum")
        top5_idx = logits.topk(5, dim=1).indices                 # [B, 5]
        top1 += (top5_idx[:, 0] == y).sum()
        top5 += (top5_idx == y[:, None]).any(dim=1).sum()
        seen += y.size(0)

        if keep_outputs:
            all_y.append(y.cpu())
            all_logits.append(logits.cpu())

    out = {
        "loss": (loss_sum / seen).item(),
        "top1": (top1.float() / seen).item(),
        "top5": (top5.float() / seen).item(),
    }
    if keep_outputs:
        out["y_true"] = torch.cat(all_y).numpy()
        out["logits"] = torch.cat(all_logits).numpy()        # [N, 101]; y_pred = logits.argmax(1)
    return out


In [ ]:
def sync(device):
    """
    Chờ GPU/MPS tính xong trước khi đọc đồng hồ.
    GPU/MPS chạy bất đồng bộ. Python gửi lệnh đi rồi chạy tiếp ngay mà không chờ GPU tính xong. 
    Nếu đọc time.time() mà không đồng bộ, sẽ đo thời gian gửi lệnh chứ không phải thời gian tính thật.
    """
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()




def memory_mb(device) -> float:
    """
    Trả về bộ nhớ tối đa đã cấp phát trên GPU/MPS tính bằng MB.
    MPS không có API trả về “đỉnh” như CUDA. 
    Cách thay thế là đọc giá trị hiện tại ở cuối mỗi epoch rồi lấy max qua các epoch.
    """
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated() / 2**20        # đỉnh từ lần reset gần nhất
    if device.type == "mps":
        return torch.mps.driver_allocated_memory() / 2**20      # hiện tại (MPS không có API đỉnh)
    return float("nan")

In [ ]:
import json, math, time
from pathlib import Path

def fit(model, loaders, cfg: dict, run_name: str, device, out_dir: Path = Path("results")):
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = Path("checkpoints"); ckpt_dir.mkdir(exist_ok=True)

    # --- optimizer / scheduler / scaler
    optimizer = torch.optim.AdamW(
        param_groups(model, cfg["arch"], cfg["lr_head"],
                     cfg["backbone_lr_mult"][cfg["init"]], cfg["weight_decay"]),
        betas=cfg["betas"], eps=cfg["eps"],
    )
    steps_per_epoch = math.ceil(len(loaders["train"]) / cfg["accum_steps"])      # 474
    scheduler = make_scheduler(optimizer,
                               total_steps=cfg["epochs"] * steps_per_epoch,
                               warmup_steps=cfg["warmup_epochs"] * steps_per_epoch)
    scaler = torch.amp.GradScaler(device.type, enabled=cfg["amp"])



    history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_top1", "val_top5",
                               "lr", "epoch_time", "val_time", "memory_mb"]}
    best = {"val_top1": -1.0, "epoch": None}
    start_epoch = 0

    # resume from checkpoint if exists
    last_path = ckpt_dir / f"{run_name}_last.pt"
    if last_path.exists():
        state = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        history, best, start_epoch = state["history"], state["best"], state["epoch"]
        torch.set_rng_state(state["rng_cpu"])
        print(f"Resumed {run_name} from epoch {start_epoch}")


    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    

    for epoch in range(start_epoch, cfg["epochs"]):
        # lr_now = optimizer.param_groups[0]["lr"]
        lr_now = next(g["lr"] for g in optimizer.param_groups if g["name"].startswith("head"))
        
        sync(device)
        t0 = time.time()

        tr = train_one_epoch(model, loaders["train"], optimizer, scheduler, scaler, device,
                             cfg["accum_steps"], cfg["grad_clip"], cfg["amp"],
                             tag=f"[{run_name}] ep {epoch + 1}/{cfg['epochs']}")
        
        sync(device)
        t1 = time.time()
        
        va = evaluate(model, loaders["val"], device, cfg["amp"])

        sync(device)
        t2 = time.time()

        for k, v in [("train_loss", tr["loss"]), ("train_acc", tr["acc"]),
                     ("val_loss", va["loss"]), ("val_top1", va["top1"]), ("val_top5", va["top5"]),
                     ("lr", lr_now), ("epoch_time", t1 - t0), ("val_time", t2 - t1),
                     ("memory_mb", memory_mb(device))]:
            history[k].append(v)


        # save checkpoint
        if va["top1"] > best["val_top1"]:
            best = {"val_top1": va["top1"], "epoch": epoch + 1}
            torch.save({"model": model.state_dict(), "epoch": epoch + 1, "config": cfg},
                       ckpt_dir / f"{run_name}_best.pt")

        # save last checkpoint
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "history": history, "best": best, "epoch": epoch + 1,
                    "rng_cpu": torch.get_rng_state()}, last_path)

        line = f"[{run_name}] epoch {epoch + 1:2d}/{cfg['epochs']} | lr {lr_now:.2e} | " \
               f"train {tr['loss']:.3f}/{tr['acc']:.3f} | val {va['loss']:.3f}/{va['top1']:.3f} " \
               f"(top5 {va['top5']:.3f}) | {t1 - t0:.0f}s + {t2 - t1:.0f}s"

        print(line, flush=True)
        with open(out_dir / f"{run_name}.log", "a") as f: f.write(line + "\n")

    summary = {"run": run_name, "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in cfg.items()},
               "best": best, "history": history,
               "total_params": sum(p.numel() for p in model.parameters()),
               "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
               "total_train_time_s": sum(history["epoch_time"]),
               "total_val_time_s": sum(history["val_time"]),
               "weights": PRETRAINED_WEIGHTS[cfg["arch"]].url if cfg["init"] == "pretrained" else None
              }

    (out_dir / f"{run_name}.json").write_text(json.dumps(summary, indent=2))
    return summary

## Kiểm tra trên một tập nhỏ, sample ít

- Để kiểm tra overfit tập nhỏ
- Đo thử thời gian và bộ nhớ chạy

In [ ]:
# overfit một tập nhỏ
from torch.utils.data import Subset, DataLoader
from data import Food101Dataset, build_transforms

train_rows = [r for r in manifest if r["split"] == "train"][:64]
tiny = Food101Dataset(train_rows, CONFIG["data_dir"], build_transforms(train=False))   # không augment
tiny_loader = DataLoader(tiny, batch_size=32, shuffle=True, num_workers=0)

for arch in ["resnet50", "vit_b_16"]:
    tiny_cfg = {**CONFIG, "epochs": 100, "warmup_epochs": 0, "lr_head": 1e-3,
                "arch": arch, "init": "scratch", "freeze": "full"}
    set_seed(CONFIG["seed"])
    model = build_model(arch, NUM_CLASSES).to(DEVICE)
    s = fit(model, {"train": tiny_loader, "val": tiny_loader}, tiny_cfg,
            run_name=f"_check_overfit_{arch}", device=DEVICE, out_dir=Path("results/_checks"))
    print(arch, "final train acc:", s["history"]["train_acc"][-1])


In [ ]:
# pilot đo thời thử thời gian và bộ nhớ
def pilot(arch, n_steps=50):
    set_seed(CONFIG["seed"])
    model = build_model(arch, NUM_CLASSES).to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=CONFIG["amp"])
    it = iter(loaders["train"])
    times = []
    for i in range(n_steps + 5):                               # 5 step đầu để warm up
        x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)
        sync(DEVICE); t0 = time.time()
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=CONFIG["amp"]):
            loss = F.cross_entropy(model(x).float(), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        sync(DEVICE)
        if i >= 5:
            times.append(time.time() - t0)
    per_batch = float(np.median(times))
    epoch_min = per_batch * len(loaders["train"]) / 60
    print(f"{arch}: {per_batch * 1000:.0f} ms/batch -> ~{epoch_min:.1f} min/epoch train, "
          f"~{epoch_min * 50 / 60:.1f} h / 50 epochs, memory {memory_mb(DEVICE):.0f} MB")

for arch in ["resnet50", "vit_b_16"]:
    pilot(arch)

## Kiểm tra trước khi train

In [ ]:
w = ResNet50_Weights.IMAGENET1K_V1
m = resnet50(weights=w).eval().to(DEVICE)
idx = next(i for i, y in enumerate(datasets["val"].labels) if CLASS_NAMES[y] == "pizza")
x, _ = datasets["val"][idx]
with torch.no_grad():
    top5 = m(x[None].to(DEVICE)).softmax(1).topk(5)
print([(w.meta["categories"][i], f"{p:.2f}") for p, i in zip(top5.values[0].tolist(), top5.indices[0].tolist())])

In [ ]:
# Kiểm tra việc freeze các module trong mô hình
# Hàm kiểm tra xem các module bị freeze có thực sự không thay đổi sau khi optimizer step hay không

def check_freeze(arch, init, freeze):
    set_seed(0)
    model = build_model(arch, NUM_CLASSES, init, head_seed=0)
    apply_freeze(model, arch, freeze)
    model = model.to(DEVICE)
    before = {k: v.detach().clone() for k, v in model.state_dict().items()}   # gồm cả buffer BN

    opt = torch.optim.AdamW(param_groups(model, arch, 1e-3, 0.1, 0.01))
    set_train_mode(model)
    x, y = next(iter(loaders["train"]))
    F.cross_entropy(model(x.to(DEVICE)), y.to(DEVICE)).backward()
    opt.step()

    after = model.state_dict()
    changed = {k for k in before if not torch.equal(before[k], after[k])}
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    print(f"{arch}/{freeze}: trainable {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    # 1) mọi tham số trainable phải thay đổi
    assert trainable <= changed, trainable - changed
    # 2) không tham số/buffer nào của phần bị freeze được thay đổi
    frozen_prefixes = tuple(m + "." for m in model.frozen_modules)
    leaked = {k for k in changed if k.startswith(frozen_prefixes)}
    assert not leaked, f"frozen state changed: {sorted(leaked)[:5]}"
    print("  OK: frozen params + BN buffers unchanged")

for arch, init, freeze in [("resnet50", "pretrained", "head"), ("resnet50", "pretrained", "partial")]:
    check_freeze(arch, init, freeze)

In [ ]:
# Tạo các tập dữ liệu nhỏ để thử nghiệm nhanh

from torch.utils.data import Subset

def tiny_datasets(n_train=256, n_val=128, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "train": Subset(datasets["train"], rng.choice(len(datasets["train"]), n_train, replace=False)),
        "val":   Subset(datasets["val"],   rng.choice(len(datasets["val"]),   n_val,   replace=False)),
    }

In [ ]:
# Đo thời gian chạy
def estimate_epoch_time(model, loaders, device, n_steps=20, warmup=3, amp=True, epochs=50):
    """Ước tính thời gian train + val cho một epoch và cho toàn bộ `epochs`, từ median của n_steps."""
    def timed(fn, loader):
        it, ts = iter(loader), []
        for i in range(warmup + n_steps):
            x, y = next(it)
            x, y = x.to(device), y.to(device)
            sync(device); t = time.time()
            fn(x, y)
            sync(device)
            if i >= warmup:
                ts.append(time.time() - t)
        return float(np.median(ts))

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    def train_step(x, y):
        set_train_mode(model)
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            loss = F.cross_entropy(model(x).float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)

    @torch.no_grad()
    def eval_step(x, y):
        model.eval()
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            model(x)

    t_train = timed(train_step, loaders["train"]) * len(loaders["train"])
    t_val = timed(eval_step, loaders["val"]) * len(loaders["val"])
    epoch_min = (t_train + t_val) / 60
    print(f"train {t_train/60:.1f} min + val {t_val/60:.1f} min = {epoch_min:.1f} min/epoch "
          f"-> {epoch_min * epochs / 60:.1f} h / {epochs} epochs")
    return epoch_min

## Chạy thí nghiệm

In [ ]:
def run_experiment(arch: str, init: str, freeze: str, seed: int, cfg: dict, ds=datasets, out_dir=Path("results")):
    cfg = {**cfg, "arch": arch, "init": init, "freeze": freeze, "seed": seed}   # lưu vào JSON/checkpoint
    set_seed(seed)
    model = build_model(arch, NUM_CLASSES, init, cfg["head_init_std"], head_seed=seed)
    frozen = apply_freeze(model, arch, freeze)
    model = model.to(DEVICE)
    cfg["frozen_modules"] = frozen

    loaders = make_loaders(ds, batch_size=cfg["physical_batch"], seed=seed,
                           num_workers=cfg["num_workers"], 
                           pin_memory=(DEVICE.type == "cuda"))
    
    run_name = f"{arch}_{init}_{freeze}_lr{cfg['lr_head']:.0e}_ep{cfg['epochs']}_seed{seed}"
    return fit(model, loaders, cfg, run_name, DEVICE, out_dir=out_dir)

In [ ]:
# Chạy thử với các tập dữ liệu nhỏ
smoke_cfg = {**CONFIG, "epochs": 2, "warmup_epochs": 1}
for arch, init, freeze in EXPERIMENTS:
    s = run_experiment(arch, init, freeze, seed=0, cfg=smoke_cfg,
                       ds=tiny_datasets(), out_dir=Path("results/_checks"))
    print(f"{arch:9s} {init:10s} {freeze:7s} trainable {s['trainable_params']:>11,} "
          f"| val top1 {s['best']['val_top1']:.3f}")

In [ ]:
# Đo thời gian thgat65 cho từng cấu hình, rồi ước tính tổng
estimates = {}
for arch, init, freeze in EXPERIMENTS:
    set_seed(0)
    m = build_model(arch, NUM_CLASSES, init)
    apply_freeze(m, arch, freeze)
    m = m.to(DEVICE)
    print(f"{arch} / {init} / {freeze}: ", end="")
    estimates[(arch, init, freeze)] = estimate_epoch_time(m, loaders, DEVICE, n_steps=20, epochs=CONFIG["epochs"])
    del m
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

total_h = sum(estimates.values()) * CONFIG["epochs"] / 60
print(f"\nTổng ước tính: {total_h:.1f} h cho {len(EXPERIMENTS)} cấu hình x {CONFIG['epochs']} epoch "
      f"(+10-20% dự phòng -> {total_h * 1.1:.1f}-{total_h * 1.2:.1f} h)")

## Runner

In [ ]:
# for arch, init, freeze in EXPERIMENTS:
#     summary = run_experiment(arch, init, freeze, seed=CONFIG["seed"], cfg=CONFIG,
#                              ds=datasets, out_dir=Path("results/experiments"))
#     print(arch, init, freeze, "->", summary["best"])

In [ ]:
import gc, traceback

RUN_ORDER = [EXPERIMENTS[i] for i in (3, 4, 1, 2, 0)]   # head, partial, R-full, ViT-full, R-scratch
OUT_DIR = Path("results/experiments")
run_start = time.time()
for k, (arch, init, freeze) in enumerate(RUN_ORDER, 1):
    print(f"\n{'=' * 80}\n[{time.strftime('%H:%M:%S')}] RUN {k}/{len(RUN_ORDER)}: {arch} / {init} / {freeze} "
          f"| elapsed {(time.time() - run_start) / 3600:.1f} h\n{'=' * 80}", flush=True)
    try:
        s = run_experiment(arch, init, freeze, seed=CONFIG["seed"], cfg=CONFIG, ds=datasets, out_dir=OUT_DIR)
        print(f"DONE {arch}/{init}/{freeze}: best val top-1 {s['best']['val_top1']:.4f} @ epoch {s['best']['epoch']}")
    except Exception:
        traceback.print_exc()                       # ghi lỗi, chuyển sang cấu hình tiếp theo
    finally:
        gc.collect()
        if DEVICE.type == "mps":
            torch.mps.empty_cache()